## 9. Leakage And Forecast-Origin Availability Assumptions

The curated schema does not contain publication dates. The metadata file below is an explicit assumption set for a one-quarter-ahead modelling workflow. It distinguishes the observation quarter, the release/publication lag, and the forecast origin, but it does not contain true historical release vintages. The project currently uses revised historical ABS/RBA/market data, so the safe-lag check is a transparent leakage-control assumption rather than a real-time data audit.

In [11]:
feature_availability = []
for feature in EXTERNAL_LAGGED_FEATURES:
    base = lag_base_name(feature)
    lag = feature_lag(feature)
    assumption = availability_assumptions.loc[availability_assumptions['variable'] == base]
    min_safe_lag = int(assumption['assumed_min_safe_lag_quarters'].iloc[0]) if not assumption.empty else np.nan
    feature_availability.append({
        'candidate_feature': feature,
        'variable': base,
        'feature_lag_quarters': lag,
        'assumed_min_safe_lag_quarters': min_safe_lag,
        'forecast_origin_safe_under_assumption': bool(pd.notna(min_safe_lag) and lag >= min_safe_lag),
    })

feature_availability = pd.DataFrame(feature_availability)
display(availability_assumptions)
display(feature_availability)

,variable,source_family,assumed_min_safe_lag_quarters,assumption_note
0,unemployment_rate,ABS labour force,1,Monthly labour data can be revised and should ...
1,cash_rate,RBA policy rate,0,Policy rate is public when set; lagged feature...
2,wage_price_index,ABS WPI,1,Quarterly WPI is typically released after CPI ...
3,wpi_growth,derived ABS WPI growth,1,Inherits WPI release lag.
4,producer_price_index,ABS PPI,1,Conservative quarterly release-lag assumption.
5,ppi_growth,derived ABS PPI growth,1,Inherits PPI release lag.
6,commodity_price_index,RBA commodity index,1,Monthly or quarterly aggregation should avoid ...
7,commodity_growth,derived RBA commodity growth,1,Inherits commodity index aggregation lag.
8,wti_price,market price,0,Market data is observable daily; lagged featur...
9,wti_growth,derived WTI growth,0,Inherits market-data availability.


,candidate_feature,variable,feature_lag_quarters,assumed_min_safe_lag_quarters,forecast_origin_safe_under_assumption
0,cash_rate_lag1,cash_rate,1,0.0,True
1,cash_rate_lag2,cash_rate,2,0.0,True
2,cash_rate_lag4,cash_rate,4,0.0,True
3,unemployment_rate_lag1,unemployment_rate,1,1.0,True
4,unemployment_rate_lag2,unemployment_rate,2,1.0,True
5,unemployment_rate_lag4,unemployment_rate,4,1.0,True
6,wpi_growth_lag1,wpi_growth,1,1.0,True
7,wpi_growth_lag2,wpi_growth,2,1.0,True
8,ppi_growth_lag1,ppi_growth,1,1.0,True
9,ppi_growth_lag2,ppi_growth,2,1.0,True


**Interpretation.** The assumed safe lag isn't uniform across sources — and that
difference is the actual mechanism behind "leakage-aware lags," not just a label. Market-observed
series (`cash_rate`, `wti_price`) get a 0-quarter minimum safe lag, since a policy rate or a
traded oil price is public the moment it's set. Administrative/survey series collected and
revised by statistical agencies (`unemployment_rate`, `wage_price_index`,
`producer_price_index`) all get a full quarter's buffer, since those can be revised after
first release and a naive same-quarter lag would risk leaking a later-revised number into a
forecast that couldn't have seen it. This table is what `src/features.py`'s lag construction
actually encodes — not a fact verified against real ABS/RBA release calendars (the notebook
is explicit that this is an assumption set, not a real-time vintage audit).